# DWM Lab 3 — Data Preprocessing, Exploration and Visualization

### Dataset: Online Retail

**Objective:**  
To perform data preprocessing, exploratory data analysis (EDA), aggregation and visualization on an online retail transaction dataset.

The dataset contains transaction-level retail information such as invoice number, product code, product description, quantity, invoice date, unit price, customer ID and country.

## 1. Import Required Libraries

We use Pandas for data manipulation and Matplotlib/Seaborn for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Load the Dataset

The Excel file is loaded into a Pandas DataFrame.

In [ ]:
file_path = "Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
df.head()

## 3. Understand the Dataset

Check the dimensions, column names, data types and basic information.

In [ ]:
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

## 4. Statistical Summary

In [ ]:
df.describe(include='all').T

## 5. Check Missing Values

Identify missing values in each column.

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
print(missing_values)

## 6. Data Preprocessing

The dataset contains cancelled invoices and transactions with non-positive quantity or price.  
For sales analysis, these records are removed.

We also convert `InvoiceDate` to datetime format and create a new `SalesAmount` column.

> **Note:** The original dataset does not contain explicit Store or Product Category columns. Therefore, `Country` is used for geographical performance analysis and `Description`/`StockCode` are used for product-level analysis.

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove cancelled invoices
df_clean = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Keep valid sales transactions
df_clean = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
].copy()

# Create total sales/revenue
df_clean['SalesAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Date-related columns
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['MonthName'] = df_clean['InvoiceDate'].dt.month_name()
df_clean['Date'] = df_clean['InvoiceDate'].dt.date

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

df_clean.head()

## 7. Check the Cleaned Dataset

Verify the cleaned data and check whether invalid values remain.

In [ ]:
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nCleaned dataset shape:", df_clean.shape)
df_clean.head()

## 8. Product-wise Sales Revenue

Calculate total sales revenue for each product description and display the top products.

In [ ]:
product_sales = (
    df_clean.groupby('Description')['SalesAmount']
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 products by sales revenue:")
display(product_sales.head(10))

plt.figure(figsize=(12, 6))
product_sales.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Products by Sales Revenue')
plt.xlabel('Sales Amount')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

## 9. Country-wise Sales Performance

Since the dataset does not contain a Store column, `Country` is used to compare sales performance across locations.

In [ ]:
country_sales = (
    df_clean.groupby('Country')['SalesAmount']
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 countries by sales revenue:")
display(country_sales.head(10))

plt.figure(figsize=(12, 6))
country_sales.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Countries by Sales Revenue')
plt.xlabel('Sales Amount')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

## 10. Daily Sales Trend

Calculate total sales for each day and visualize the sales trend.

In [ ]:
daily_sales = (
    df_clean.groupby('Date')['SalesAmount']
    .sum()
)

plt.figure(figsize=(14, 6))
daily_sales.plot()
plt.title('Daily Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales Amount')
plt.tight_layout()
plt.show()

## 11. Product-wise Quantity Sold

Find the products with the highest total quantity sold.

In [ ]:
product_quantity = (
    df_clean.groupby('Description')['Quantity']
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 products by quantity sold:")
display(product_quantity.head(10))

plt.figure(figsize=(12, 6))
product_quantity.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Products by Quantity Sold')
plt.xlabel('Quantity')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

## 12. Country-wise Monthly Sales

Create a monthly sales summary to observe changes in revenue over time.

In [ ]:
monthly_sales = (
    df_clean.groupby(['Year', 'Month'])['SalesAmount']
    .sum()
    .reset_index()
)

monthly_sales['Period'] = (
    monthly_sales['Year'].astype(str) + '-' +
    monthly_sales['Month'].astype(str).str.zfill(2)
)

display(monthly_sales)

plt.figure(figsize=(12, 6))
plt.plot(monthly_sales['Period'], monthly_sales['SalesAmount'], marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Quantity vs Sales Amount

Study the relationship between quantity sold and sales revenue.

In [ ]:
sample_df = df_clean.sample(
    min(5000, len(df_clean)),
    random_state=42
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sample_df,
    x='Quantity',
    y='SalesAmount',
    alpha=0.5
)
plt.title('Quantity vs Sales Amount')
plt.xlabel('Quantity')
plt.ylabel('Sales Amount')
plt.tight_layout()
plt.show()

## 14. Distribution of Sales Amount

Use a boxplot to identify the distribution and possible outliers in transaction sales.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df_clean['SalesAmount'])
plt.title('Distribution of Sales Amount')
plt.xlabel('Sales Amount')
plt.tight_layout()
plt.show()

## 15. Customer-wise Sales

Calculate total revenue generated by customers.

Only records with a valid `CustomerID` are considered.

In [ ]:
customer_sales = (
    df_clean.dropna(subset=['CustomerID'])
    .groupby('CustomerID')['SalesAmount']
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 customers by sales:")
display(customer_sales.head(10))

## 16. Summary of Important Findings

The analysis can be used to identify:

- Top-selling products
- Countries generating the highest revenue
- Daily and monthly sales trends
- Products with the highest quantities sold
- Customer-wise purchasing contribution
- Distribution and outliers in sales transactions
- Relationship between quantity and sales amount

## Conclusion

The Online Retail dataset was successfully loaded, cleaned and explored.  
Different aggregation and visualization techniques were applied to understand product sales, geographical performance, customer contribution and sales trends.

The cleaned dataset and the derived `SalesAmount` field can also be reused for the upcoming DWM experiments involving regression, classification, clustering and association rule mining.